# Font Recognition — Validation Report (Colab)

Runtime → Change runtime type → **GPU**.

Uses the checkpoint in `out` (from `train_colab.ipynb`), predicts the top-10 fonts for every image in every `val-p*.npz` shard, then reports how often the correct font is in the top 10 and in the top 5, with precision / recall / F1 and per-font breakdowns.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

## 2. Imports

In [ ]:
import os, csv, glob, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn

## 3. Settings

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    data = '/content/drive/MyDrive/fontdataset',                       # val-p*.npz, fonts.csv, chars.json
    out  = '/content/drive/MyDrive/FontDetection/runs/font_arcface',   # best.pt / last.pt / font_index.npz
    val_dir = '/content/drive/MyDrive/fontdataset_val_clean',          # val-p*.npz to test (None = the original val in data)
    ckpt = 'best.pt',             # falls back to last.pt
    methods = ['head', 'proto'],  # 'head' = ArcFace classifier, 'proto' = font_index.npz (what predict() uses)
    kmax = 10,                    # fonts kept per image
    batch = 1024,
    val_limit = None,             # only the first N val shards (None = all)
    reuse = True,                 # reuse eval/predictions.npz if it matches the checkpoint + val shards
    show_examples = 10,           # images printed with their top-10 list
    worst = 15,                   # rows in the worst fonts / chars / confusions tables
    save_per_image = True,        # CSV with the top-10 fonts of every val image
)

args.val_dir = (args.val_dir or args.data).rstrip('/')
args.eval_dir = os.path.join(args.out, 'eval', os.path.basename(args.val_dir))
os.makedirs(args.eval_dir, exist_ok=True)
val_files = sorted(glob.glob(os.path.join(args.val_dir, 'val-p*.npz')))[:args.val_limit]
print(len(val_files), 'val shards from', args.val_dir, '| results ->', args.eval_dir)

## 4. Model

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
                   if stride != 1 or cin != cout else nn.Identity())

    def forward(self, x):
        y = Fn.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return Fn.relu(y + self.sc(x))

class FontEncoder(nn.Module):
    def __init__(self, emb_dim=256, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(1, w // 2, 3, 1, 1, bias=False), nn.BatchNorm2d(w // 2),
                                  nn.ReLU(),
                                  nn.Conv2d(w // 2, w, 3, 2, 1, bias=False), nn.BatchNorm2d(w),
                                  nn.ReLU())
        self.layers = nn.Sequential(
            Block(w, w, 1),         Block(w, w, 1),          # 32x32
            Block(w, 2 * w, 2),     Block(2 * w, 2 * w, 1),  # 16x16
            Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1),  # 8x8
            Block(4 * w, 8 * w, 2), Block(8 * w, 8 * w, 1),  # 4x4
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.1), nn.Linear(8 * w * 2, emb_dim),
                                  nn.BatchNorm1d(emb_dim))

    def forward(self, x):
        h = self.layers(self.stem(x))
        h = torch.cat([Fn.adaptive_avg_pool2d(h, 1), Fn.adaptive_max_pool2d(h, 1)], 1)
        return Fn.normalize(self.head(h), dim=1)

def to_input(x_uint8, device):
    # (B,64,64) uint8 -> (B,1,64,64) float in [-1,1]
    return (x_uint8.to(device, non_blocking=True).float().unsqueeze(1) / 127.5 - 1.0)

def pick_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_styles(data_dir):
    with open(os.path.join(data_dir, "fonts.csv"), newline="", encoding="utf-8") as fh:
        return [(r["family"], r["style"]) for r in csv.DictReader(fh)]

## 5. Load checkpoint + font index

In [ ]:
device = pick_device()
ck_path = os.path.join(args.out, args.ckpt)
if not os.path.exists(ck_path): ck_path = os.path.join(args.out, 'last.pt')
ck = torch.load(ck_path, map_location=device, weights_only=True)
cfg = ck['cfg']
model = FontEncoder(cfg['emb_dim'], cfg['width']).to(device)
model.load_state_dict(ck['model']); model.eval()

styles = load_styles(args.data)
n_cls = len(styles)
font_name = np.array([f'{f} {s}' for f, s in styles])
family_name = np.array([f for f, _ in styles])
fam_id = np.unique(family_name, return_inverse=True)[1]
with open(os.path.join(args.data, 'chars.json'), encoding='utf-8') as fh:
    chars = json.load(fh)['chars']
char_name = lambda c: chars[c] if c < len(chars) else str(c)

# each scorer: (unit-norm class matrix [n_classes, emb_dim], mask of classes that can't be predicted)
scorers = {}
if 'head' in args.methods:
    W = ck['head']['W'].float()
    W = torch.cat([W, torch.zeros(n_cls - len(W), W.shape[1], device=W.device)]) if len(W) < n_cls else W
    scorers['head'] = (Fn.normalize(W, dim=1).to(device), (W.abs().sum(1) == 0).to(device))
if 'proto' in args.methods:
    idx_path = os.path.join(args.out, 'font_index.npz')
    if os.path.exists(idx_path):
        if os.path.getmtime(idx_path) < os.path.getmtime(ck_path):
            print('warning: font_index.npz is older than the checkpoint — re-run "Build font index" in train_colab.ipynb')
        idx = np.load(idx_path)
        scorers['proto'] = (torch.from_numpy(idx['prototypes']).float().to(device),
                            torch.from_numpy(~idx['valid']).to(device))
    else:
        print('font_index.npz not found — run "Build font index" in train_colab.ipynb; skipping proto')

METHOD_NAME = {'head': 'ArcFace classifier', 'proto': 'prototype index (font_index.npz)'}
print(f'device {device} | {os.path.basename(ck_path)} | {n_cls} fonts, {len(np.unique(fam_id))} families '
      f'| methods: {", ".join(scorers)}')

## 6. Predict every val image

In [ ]:
@torch.no_grad()
def predict_val(files):
    pred = {m: dict(top=[], score=[], rank=[]) for m in scorers}
    true, char, shard, pos = [], [], [], []
    t0 = time.time()
    for si, f in enumerate(files):
        with np.load(f) as d:
            X, F, C = d['x'], d['font'].astype(np.int64), d['char'].astype(np.int64)
        for i in range(0, len(F), args.batch):
            e = model(to_input(torch.from_numpy(X[i:i + args.batch]), device))
            y = torch.from_numpy(F[i:i + args.batch]).to(device)
            for m, (P, blocked) in scorers.items():
                s = e @ P.t()
                s[:, blocked] = -2
                rank = (s > s.gather(1, y[:, None])).sum(1) + 1      # 1 = correct font scored highest
                v, ix = s.topk(args.kmax, 1)
                pred[m]['top'].append(ix.int().cpu().numpy())
                pred[m]['score'].append(v.cpu().numpy())
                pred[m]['rank'].append(rank.int().cpu().numpy())
        true.append(F); char.append(C)
        shard.append(np.full(len(F), si, np.int16)); pos.append(np.arange(len(F), dtype=np.int32))
        run = ' | '.join(f'{m} ' + ' '.join(f'top{k} {np.mean(np.concatenate(pred[m]["rank"]) <= k):.4f}'
                                             for k in (1, 5, 10) if k <= args.kmax) for m in pred)
        print(f'[{si + 1}/{len(files)}] {os.path.basename(f)} {len(F)} imgs | {(time.time() - t0) / 60:.1f}m '
              f'| running {run}', flush=True)
    R = dict(true=np.concatenate(true), char=np.concatenate(char),
             shard=np.concatenate(shard), pos=np.concatenate(pos),
             files=[os.path.basename(f) for f in files],
             pred={m: {k: np.concatenate(v) for k, v in p.items()} for m, p in pred.items()})
    return R

def save_preds(R, path):
    flat = {f'{m}_{k}': v for m, p in R['pred'].items() for k, v in p.items()}
    meta = dict(ckpt=ck_path, ckpt_mtime=os.path.getmtime(ck_path), val_dir=args.val_dir, files=R['files'],
                methods=list(R['pred']), kmax=args.kmax)
    np.savez_compressed(path, true=R['true'], char=R['char'], shard=R['shard'], pos=R['pos'],
                        meta=json.dumps(meta), **flat)

def load_preds(path):
    if not os.path.exists(path): return None
    d = np.load(path)
    meta = json.loads(str(d['meta']))
    if (meta['ckpt'] != ck_path or meta['ckpt_mtime'] != os.path.getmtime(ck_path)
            or meta.get('val_dir') != args.val_dir
            or meta['files'] != [os.path.basename(f) for f in val_files]
            or not set(scorers) <= set(meta['methods']) or meta['kmax'] < args.kmax):
        return None
    return dict(true=d['true'], char=d['char'], shard=d['shard'], pos=d['pos'], files=meta['files'],
                pred={m: {k: d[f'{m}_{k}'][:, :args.kmax] if k != 'rank' else d[f'{m}_{k}']
                          for k in ('top', 'score', 'rank')} for m in scorers})

## 7. Examples

In [ ]:
def show_examples(R, method, n, seed=0):
    p = R['pred'][method]
    rng = np.random.default_rng(seed)
    print(f'===== {n} random val images — {METHOD_NAME[method]} =====')
    for j in rng.choice(len(R['true']), size=min(n, len(R['true'])), replace=False):
        y, r = R['true'][j], int(p['rank'][j])
        verdict = f'rank {r}' + ('  ✓ top-10' if r <= 10 else '  ✗ top-10') + ('  ✓ top-5' if r <= 5 else '  ✗ top-5')
        print(f"\n{R['files'][R['shard'][j]]}[{R['pos'][j]}]  char '{char_name(R['char'][j])}'  "
              f"true: {font_name[y]}  →  {verdict}")
        for i, (c, s) in enumerate(zip(p['top'][j], p['score'][j]), 1):
            print(f"   {i:2d}. {font_name[c]:<45s} {s:.3f}{'   ← correct' if c == y else ''}")

## 8. Metrics

For top-**k**, an image counts as **correct** when the true font is anywhere in its top-k list. Its predicted label is then the true font; otherwise it is the top-1 font. Precision / recall / F1 are computed per font on those labels:

- **recall** (per font) = share of that font's val images whose top-k list contains it (its top-k hit rate)
- **precision** (per font) = of the images labelled as this font, share that really are this font
- **macro** = plain average over fonts · **weighted** = weighted by number of val images · **micro** = over all images (equals top-k accuracy)
- **family accuracy** = a font of the correct family is in the top-k (style may be wrong)
- **MRR@k** = mean of 1/rank of the true font (0 when outside top-k) · **mean / median rank** use the full ranking over all fonts

In [ ]:
def topk_report(R, method, k):
    p = R['pred'][method]
    y, rank, top = R['true'], p['rank'], p['top'][:, :k]
    n = len(y)
    hit = rank <= k
    pred = np.where(hit, y, top[:, 0])
    support = np.bincount(y, minlength=n_cls)
    tp = np.bincount(y[hit], minlength=n_cls)
    npred = np.bincount(pred, minlength=n_cls)
    prec = np.divide(tp, npred, out=np.zeros(n_cls), where=npred > 0)
    rec = np.divide(tp, support, out=np.zeros(n_cls), where=support > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(n_cls), where=(prec + rec) > 0)
    ev = support > 0
    w = support[ev] / n
    fam_hit = (fam_id[top] == fam_id[y][:, None]).any(1)

    rep = dict(
        method=method, k=k, images=n, fonts_evaluated=int(ev.sum()),
        correct=int(hit.sum()), missed=int(n - hit.sum()),
        accuracy=hit.mean(), top1_accuracy=(rank == 1).mean(), family_accuracy=fam_hit.mean(),
        precision_macro=prec[ev].mean(), recall_macro=rec[ev].mean(), f1_macro=f1[ev].mean(),
        precision_weighted=(prec[ev] * w).sum(), recall_weighted=(rec[ev] * w).sum(), f1_weighted=(f1[ev] * w).sum(),
        precision_micro=hit.mean(), recall_micro=hit.mean(), f1_micro=hit.mean(),
        mrr=np.where(hit, 1.0 / rank, 0.0).mean(),
        mean_rank=rank.mean(), median_rank=float(np.median(rank)),
        fonts_always_correct=int((rec[ev] == 1).sum()), fonts_never_correct=int((rec[ev] == 0).sum()),
        fonts_below_50pct=int((rec[ev] < 0.5).sum()),
    )
    rep = {k_: (float(v) if isinstance(v, (np.floating, float)) else v) for k_, v in rep.items()}

    miss = ~hit
    pair = y[miss].astype(np.int64) * n_cls + top[miss, 0]
    pv, pc = np.unique(pair, return_counts=True)
    confused = np.full(n_cls, -1); confused_n = np.zeros(n_cls, int)
    for v, c in sorted(zip(pv, pc), key=lambda t: t[1]):      # keep the most frequent per true font
        confused[v // n_cls], confused_n[v // n_cls] = v % n_cls, c
    fonts = pd.DataFrame(dict(
        label_id=np.arange(n_cls), font=font_name, family=family_name, images=support, correct=tp,
        recall=rec, precision=prec, f1=f1,
        most_confused_with=[font_name[c] if c >= 0 else '' for c in confused], confused_count=confused_n,
    ))[ev].sort_values(['f1', 'images'], ascending=[True, False])

    ch = R['char']
    chars_df = pd.DataFrame(dict(char=[char_name(c) for c in range(ch.max() + 1)],
                                 images=np.bincount(ch), correct=np.bincount(ch[hit], minlength=ch.max() + 1)))
    chars_df = chars_df[chars_df.images > 0].assign(accuracy=lambda d: d.correct / d.images).sort_values('accuracy')

    confusions = pd.DataFrame(dict(true=font_name[pv // n_cls], predicted=font_name[pv % n_cls], count=pc)) \
        .sort_values('count', ascending=False)
    return rep, fonts, chars_df, confusions

def print_report(rep):
    k, pct = rep['k'], lambda v: f'{100 * v:6.2f}%'
    print(f"\n{'=' * 78}\n TOP-{k} REPORT — {METHOD_NAME[rep['method']]}\n{'=' * 78}")
    print(f" images evaluated        {rep['images']:>10,}      fonts evaluated   {rep['fonts_evaluated']:,}")
    print(f" correct font in top-{k:<3d} {rep['correct']:>10,}  ({pct(rep['accuracy']).strip()})")
    print(f" missed                  {rep['missed']:>10,}  ({pct(1 - rep['accuracy']).strip()})")
    print()
    print(f" top-{k} accuracy        {pct(rep['accuracy'])}")
    print(f" top-1 accuracy         {pct(rep['top1_accuracy'])}")
    print(f" family in top-{k:<3d}     {pct(rep['family_accuracy'])}")
    print(f" MRR@{k:<3d}               {rep['mrr']:.4f}")
    print(f" mean / median rank     {rep['mean_rank']:.2f} / {rep['median_rank']:.0f}")
    print()
    print(f" {'':<12s}{'precision':>11s}{'recall':>10s}{'f1':>10s}")
    for avg in ('macro', 'weighted', 'micro'):
        print(f" {avg:<12s}{rep['precision_' + avg]:>11.4f}{rep['recall_' + avg]:>10.4f}{rep['f1_' + avg]:>10.4f}")
    print()
    print(f" fonts always correct   {rep['fonts_always_correct']:,} / {rep['fonts_evaluated']:,}")
    print(f" fonts below 50%        {rep['fonts_below_50pct']:,}")
    print(f" fonts never correct    {rep['fonts_never_correct']:,}")

def run_reports(R, k):
    reports = {}
    for m in R['pred']:
        rep, fonts, chars_df, confusions = topk_report(R, m, k)
        print_report(rep)
        n = args.worst
        print(f'\n worst {n} fonts (top-{k}):')
        print(fonts.head(n)[['font', 'images', 'correct', 'recall', 'precision', 'f1', 'most_confused_with']]
              .to_string(index=False, float_format='%.3f'))
        print(f'\n hardest {n} characters (top-{k}):')
        print(chars_df.head(n).to_string(index=False, float_format='%.3f'))
        print(f'\n most common misses (true → top-1 prediction):')
        print(confusions.head(n).to_string(index=False))
        tag = f'{m}_top{k}'
        with open(os.path.join(args.eval_dir, f'report_{tag}.json'), 'w') as fh: json.dump(rep, fh, indent=1)
        fonts.to_csv(os.path.join(args.eval_dir, f'per_font_{tag}.csv'), index=False)
        chars_df.to_csv(os.path.join(args.eval_dir, f'per_char_{tag}.csv'), index=False)
        confusions.to_csv(os.path.join(args.eval_dir, f'confusions_{tag}.csv'), index=False)
        reports[m] = rep
    print(f'\nsaved report_*_top{k}.json, per_font_*, per_char_*, confusions_* to {args.eval_dir}')
    return reports

def save_per_image(R):
    for m, p in R['pred'].items():
        path = os.path.join(args.eval_dir, f'per_image_{m}.csv')
        df = pd.DataFrame(dict(shard=np.array(R['files'])[R['shard']], index=R['pos'],
                               char=[char_name(c) for c in R['char']], true_font=font_name[R['true']],
                               rank=p['rank'], in_top5=p['rank'] <= 5, in_top10=p['rank'] <= 10))
        for i in range(p['top'].shape[1]):
            df[f'top{i + 1}'] = font_name[p['top'][:, i]]
            df[f'score{i + 1}'] = p['score'][:, i].round(4)
        df.to_csv(path, index=False)
        print(f'saved {path} ({len(df):,} rows)')

## 9. Run predictions

In [ ]:
pred_path = os.path.join(args.eval_dir, 'predictions.npz')
R = load_preds(pred_path) if args.reuse else None
if R is not None:
    print(f'reusing {pred_path} ({len(R["true"]):,} images)')
else:
    R = predict_val(val_files)
    save_preds(R, pred_path)
    print(f'saved {pred_path}')
if args.save_per_image:
    save_per_image(R)

## 10. Examples

In [ ]:
for m in R['pred']:
    show_examples(R, m, args.show_examples)

## 11. Top-10 report

In [ ]:
rep10 = run_reports(R, 10)

## 12. Top-5 report

In [ ]:
rep5 = run_reports(R, 5)

## 13. Summary

In [ ]:
rows = {f'{m} top-{r["k"]}': r for reps in (rep10, rep5) for m, r in reps.items()}
cols = ['accuracy', 'top1_accuracy', 'family_accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
        'precision_weighted', 'recall_weighted', 'f1_weighted', 'mrr', 'mean_rank', 'median_rank',
        'fonts_always_correct', 'fonts_below_50pct', 'fonts_never_correct']
summary = pd.DataFrame(rows).T[cols]
summary.to_csv(os.path.join(args.eval_dir, 'summary.csv'))
summary.T

## 14. Look at misses

In [ ]:
import matplotlib.pyplot as plt

def show_misses(R, method, k=5, n=24, seed=0):
    p = R['pred'][method]
    miss = np.flatnonzero(p['rank'] > k)
    if not len(miss): print('no misses'); return
    pick = np.random.default_rng(seed).choice(miss, size=min(n, len(miss)), replace=False)
    cols = 6; rows = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.4 * rows))
    cache = {}
    for ax, j in zip(np.ravel(axes), pick):
        f = R['files'][R['shard'][j]]
        if f not in cache:
            with np.load(os.path.join(args.val_dir, f)) as d: cache[f] = d['x']
        ax.imshow(cache[f][R['pos'][j]], cmap='gray', vmin=0, vmax=255)
        ax.set_title(f"true: {font_name[R['true'][j]]}\npred: {font_name[p['top'][j, 0]]}\nrank {p['rank'][j]}",
                     fontsize=8)
    for ax in np.ravel(axes): ax.axis('off')
    fig.suptitle(f'{METHOD_NAME[method]} — images whose font is not in the top {k}')
    plt.tight_layout(); plt.show()

for m in R['pred']:
    show_misses(R, m, k=5)